# Task 6 results: how many TGLF runs, and how large a network

Reads the committed tables in `results/`. All errors compare with numerical TGLF for shot 128913 and are in gyro-Bohm units. Spreads across training draws are not prediction intervals.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.precision", 3)
HERE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "notebooks/ryan").is_dir()) / "notebooks/ryan/task6"
results = HERE / "results"

## Learning curves

Mean RMSE over draws at each training-set size.

In [2]:
curves = pd.read_csv(results / "learning_curves_by_cohort.csv")
curves.pivot_table(index=["target", "cohort"], columns="size", values="mean_rmse")

size                           1000    2000    4000    8000    16586
target  cohort                                                      
Gamma_e intermediate_profile   0.127   0.074   0.070   0.096   0.078
        intermediate_varied    0.338   0.321   0.298   0.270   0.269
        nbasis                 0.439   0.301   0.311   0.341   0.307
        test                   0.412   0.338   0.307   0.326   0.278
        train                  0.194   0.170   0.166   0.163   0.147
        validation             0.382   0.303   0.265   0.224   0.171
Q_e     intermediate_profile   2.275   0.761   0.739   0.685   1.001
        intermediate_varied   17.000  14.996  13.869  12.684  12.431
        nbasis                28.751  16.992  15.367  15.552  11.508
        test                  16.682  12.498  12.065  11.137  10.594
        train                 13.143   9.149   7.892   8.274   7.480
        validation            18.554  12.809  11.235   9.982   8.666
Q_i     intermediate_profile   1.153   1.016   0.980   0.978   0.915
        intermediate_varied    1.369   1.213   1.130   1.084   1.016
        nbasis                 1.573   1.170   1.021   1.040   1.130
        test                   1.829   1.509   1.280   1.063   0.919
        train                  0.804   0.665   0.605   0.570   0.571
        validation             1.629   1.246   0.996   0.709   0.587

## Runs needed to halve the error

Power-law fits, RMSE ∝ n^(−b). These are order-of-magnitude estimates.

In [3]:
fits = pd.read_csv(results / "scaling_fits.csv")
fits.loc[fits.curve.isin(["validation", "test", "intermediate_varied"]), ["target", "curve", "window", "exponent", "r2", "runs_to_halve"]]

,target,curve,window,exponent,r2,runs_to_halve
0,Q_e,validation,all sizes (1k-16.6k),0.253,0.929,2.578e+05
1,Q_e,validation,4k-16.6k,0.183,0.999,7.380e+05
2,Q_e,test,all sizes (1k-16.6k),0.146,0.833,1.926e+06
3,Q_e,test,4k-16.6k,0.091,0.978,3.318e+07
4,Q_e,intermediate_varied,all sizes (1k-16.6k),0.113,0.948,7.632e+06
5,Q_e,intermediate_varied,4k-16.6k,0.077,0.873,1.428e+08
6,Q_i,validation,all sizes (1k-16.6k),0.372,0.993,1.070e+05
7,Q_i,validation,4k-16.6k,0.371,0.969,1.072e+05
8,Q_i,test,all sizes (1k-16.6k),0.246,0.997,2.765e+05
9,Q_i,test,4k-16.6k,0.233,0.993,3.267e+05


## Network width

Mean training and validation RMSE by hidden width (validation only; 4× width added after the freeze).

In [4]:
pd.read_csv(results / "width_capacity.csv").pivot_table(index=["target", "size", "width"], values=["hidden_width", "train_rmse", "validation_rmse"])

hidden_width  train_rmse  validation_rmse
target  size  width                                              
Gamma_e 1000  narrow            32.0       0.274            0.431
              original          64.0       0.194            0.382
              wide             128.0       0.140            0.371
        2000  original          64.0       0.170            0.303
        4000  narrow            32.0       0.190            0.285
              original          64.0       0.166            0.265
              wide             128.0       0.133            0.262
        8000  original          64.0       0.163            0.224
        16586 narrow            32.0       0.185            0.204
              original          64.0       0.147            0.171
              wide             128.0       0.155            0.185
              xwide            256.0       0.171            0.188
Q_e     1000  narrow            32.0      14.159           18.863
              original          64.0      13.143           18.554
              wide             128.0      13.049           19.036
        2000  original          64.0       9.149           12.809
        4000  narrow            32.0       8.840           11.698
              original          64.0       7.892           11.235
              wide             128.0       7.496           11.063
        8000  original          64.0       8.274            9.982
        16586 narrow            32.0       7.919            9.352
              original          64.0       7.480            8.666
              wide             128.0       6.317            7.837
              xwide            256.0       5.832            7.499
Q_i     1000  narrow            64.0       0.878            1.636
              original         128.0       0.804            1.629
              wide             256.0       0.760            1.643
        2000  original         128.0       0.665            1.246
        4000  narrow            64.0       0.810            1.093
              original         128.0       0.605            0.996
              wide             256.0       0.533            1.006
        8000  original         128.0       0.570            0.709
        16586 narrow            64.0       0.601            0.593
              original         128.0       0.571            0.587
              wide             256.0       0.545            0.582
              xwide            512.0       0.558            0.594

## Error and 90% interval coverage by radius (full training set)

In [5]:
radius = pd.read_csv(results / "error_by_radius.csv")
radius.loc[radius.cohort.isin(["test", "intermediate_varied"])].pivot_table(index="r_over_a", columns=["target", "cohort"], values="coverage")

target               Gamma_e                        Q_e         \
cohort   intermediate_varied   test intermediate_varied   test   
r_over_a                                                         
0.200                    NaN  0.964                 NaN  0.982   
0.225                  0.950    NaN               0.983    NaN   
0.250                    NaN  0.909                 NaN  0.982   
0.275                  0.900    NaN               0.983    NaN   
0.300                    NaN  0.859                 NaN  0.964   
0.325                  0.983    NaN               0.917    NaN   
0.350                    NaN  0.942                 NaN  0.942   
0.375                  0.508    NaN               0.367    NaN   
0.400                    NaN  0.935                 NaN  0.935   
0.425                  0.933    NaN               0.967    NaN   
0.450                    NaN  0.924                 NaN  0.975   
0.475                  0.942    NaN               0.992    NaN   
0.500                    NaN  0.949                 NaN  0.975   
0.525                  0.850    NaN               0.950    NaN   
0.550                    NaN  0.855                 NaN  0.909   
0.575                  0.866    NaN               0.958    NaN   
0.600                    NaN  0.927                 NaN  0.931   
0.625                  0.900    NaN               0.925    NaN   
0.650                    NaN  0.857                 NaN  0.890   
0.675                  0.795    NaN               0.872    NaN   
0.700                    NaN  0.865                 NaN  0.809   
0.725                  0.863    NaN               0.812    NaN   
0.750                    NaN  0.831                 NaN  0.733   
0.775                  0.844    NaN               0.624    NaN   
0.800                    NaN  0.892                 NaN  0.714   

target                   Q_i         
cohort   intermediate_varied   test  
r_over_a                             
0.200                    NaN  0.967  
0.225                  1.000    NaN  
0.250                    NaN  0.975  
0.275                  0.908    NaN  
0.300                    NaN  0.884  
0.325                  0.917    NaN  
0.350                    NaN  0.898  
0.375                  0.392    NaN  
0.400                    NaN  0.859  
0.425                  0.925    NaN  
0.450                    NaN  0.957  
0.475                  0.975    NaN  
0.500                    NaN  0.960  
0.525                  0.858    NaN  
0.550                    NaN  0.865  
0.575                  0.807    NaN  
0.600                    NaN  0.898  
0.625                  0.925    NaN  
0.650                    NaN  0.890  
0.675                  0.906    NaN  
0.700                    NaN  0.903  
0.725                  0.880    NaN  
0.750                    NaN  0.793  
0.775                  0.615    NaN  
0.800                    NaN  0.781

## Surrogate error compared with TGLF's own resolution sensitivity

Differences from TGLF at NBASIS 12/12 on 385 cases.

In [6]:
pd.read_csv(results / "tglf_resolution.csv")

,target,comparison,n,rmse,median_relative_difference,share_within_5pct
0,Q_e,surrogate (full-data A6 model),385,11.508,0.024,0.740
1,Q_e,"TGLF, NBASIS 16/16",385,23.087,0.062,0.421
2,Q_e,"TGLF, NBASIS 2/4",385,24.891,0.239,0.127
3,Q_i,surrogate (full-data A6 model),385,1.130,0.022,0.706
4,Q_i,"TGLF, NBASIS 16/16",385,1.926,0.052,0.486
5,Q_i,"TGLF, NBASIS 2/4",385,8.170,0.310,0.138
6,Gamma_e,surrogate (full-data A6 model),385,0.307,0.047,0.514
7,Gamma_e,"TGLF, NBASIS 16/16",385,0.499,0.120,0.244
8,Gamma_e,"TGLF, NBASIS 2/4",385,0.862,0.558,0.036
